# analyzeData

In [1]:
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
def scanFpFile(data):
    samples = []
        
    rawData = []
    normalizedData = []
    setNames = []
    
    excludeSamples = ["BS94", "BS91", "BS64", "22s", "30s"]
    
    for sample in data["samples"]:
        if not sample["name"] in (excludeSamples):
            setNames.append(sample["name"])
            rDat = []
            nDat = []
            for spots in sample["data"]:
                rDat.append(float(spots["rawIntensity"]))
                nDat.append(float(spots["normalizedIntensity"]))
            rawData.append(rDat.copy())
            normalizedData.append(nDat.copy())
    sampleNumber = 0
    #sequence = ["E","E","N","L","K","T","A","N","C","T","L","A","N","T","N","Y"]
    sequence = ["C","T","L","A","N","T","N","Y","V","N","I","T","K","M","L","T"]
    fpTables = []
    for counter, sample in enumerate(rawData):
        table,colNames,rowNames = singleSampleTable(sample,setNames[counter], sequence)
        fpTables.append(table)

    #printHeatmap("KCNA2-fp", normalizedData, spotNames, setNames)
    printMultiMap(fpTables,setNames, colNames, rowNames)

In [3]:
def printFPHeatmap(name, data, spotNames, setNames):
    
    plt.subplots(figsize=(40,20))
    df = pd.DataFrame(data, columns=spotNames, index = setNames)
    
    p1 = sns.heatmap(df ,cmap="vlag",center=1, cbar=False) # Greys
    
    p1.set_yticklabels(p1.get_ymajorticklabels(), fontsize = 18, rotation=0)
    p1.set_xticklabels(p1.get_xmajorticklabels(), fontsize = 18, rotation=0)
    cbar = p1.collections[0].colorbar
    cbar.ax.tick_params(labelsize=18)
    plt.title(""+name, fontsize = 10)
    
    figure = p1.get_figure()    
    figure.savefig("C:/Users/AGANDO/Desktop/plots/"+name +" - fp multi.png", dpi=100)

In [4]:
def indexToPosition (index):
    return ""+chr(int(index/24)+65) + str(index%24 + 1)

In [5]:
def singleSampleTable(rawData, name, colNames):
    i = 0
    rData = rawData [6:]
    
    rowNames = ["A","C","D","E","F","G","H","I","K","L","M","N","P","Q","R","S","T","V","W","Y"]
    
    data = [ [-1]*len(colNames) for i in range(len(rowNames))]
    
    for count, dat in enumerate(rData):
        data[count % len(rowNames)][int(count / len(rowNames))] = dat
    
    avgSeq = 0
    c = 0
    for col, colName in enumerate(colNames):
        for row, rowName in enumerate(rowNames):
            if colName == rowName:
                avgSeq += data[row][col]
                c += 1
    avgSeq /=c
    
    for col, colName in enumerate(colNames):
        for row, rowName in enumerate(rowNames):
            data[row][col] /= avgSeq
    
    return data, colNames, rowNames

In [6]:
def positionToIndex (position):
    letter = (ord(position[0]) - 65)*24
    number = int(position[1:]) -1
    
    return letter + number

In [7]:
def colShowcase(data, sampleNames, spotNames, setNames):
    
    rows = 4
    cols = 4
    colProfile = "vlag"
    colProfiles = ["vlag", "PiYG", "PRGn", "BrBG", "PuOr", "RdGy", "RdBu", "RdYlBu", "RdYlGn", "Spectral", "coolwarm", "bwr", "seismic", "icefire"]
    
    figWidth = len(spotNames)
    figHeight = len(setNames)
    if figWidth > figHeight:
        figHeight = figWidth

    fig, axs = plt.subplots(nrows = rows, ncols = cols, figsize=(figHeight, figHeight))
    #plt.suptitle("KCNA2 - full positional", fontsize = 50) 
    plots = []
    dataFrames = []
    for counter, profile in enumerate(colProfiles):
        
        
        for i, dat in enumerate(data):
            dataFrames.append(pd.DataFrame(dat.copy(), columns=spotNames.copy(), index = setNames.copy()))

            if rows == 1 or cols == 1:
                if rows == cols:
                    plots.append(sns.heatmap(dataFrames[counter] , ax= axs, cmap=profile ,center=1, cbar=False, square = True))
                else:
                    plots.append(sns.heatmap(dataFrames[counter] , ax= axs[counter], cmap=colProfile,center=1, cbar=False, square = True))#,linewidths= 4,linecolor = "black"))
            else:
                plots.append(sns.heatmap(dataFrames[i] , ax= axs[int(counter / cols), counter % cols], cmap=profile,center=1, cbar=False)) #vlag
            
            plots[counter].set_yticklabels(plots[counter].get_ymajorticklabels(), fontsize = 10, rotation=0)
            plots[counter].set_xticklabels(plots[counter].get_xmajorticklabels(), fontsize = 10, rotation=0)

            #plots[counter].set_title(""+profile, fontsize = 30)

        #fig.delaxes(axs[-1][-1])
        #resolutions = [50,100,150,200]
        resolutions = [100]
        for res in resolutions:
            plt.savefig("C:/Users/KaiKr/Desktop/plots/KCNA2 - fp - colProfiles - "+str(res)+" dpi.png", dpi=res)

In [24]:
def printMultiMap(data, sampleNames, spotNames, setNames):
    
    rows = 1
    cols = 2
    colProfile = "vlag"
    
    figWidth = len(spotNames)
    figHeight = len(setNames)
    if figWidth > figHeight:
        figHeight = figWidth

    fig, axs = plt.subplots(nrows = rows, ncols = cols, figsize=(220, 100))
   
    plt.suptitle("KCNA2 - full positional - 111", fontsize = 250) 
    plots = []
    dataFrames = []
    
    for counter, dat in enumerate(data):
        dataFrames.append(pd.DataFrame(dat.copy(), columns=spotNames.copy(), index = setNames.copy()))

        if rows == 1 or cols == 1:
            if rows == cols:
                plots.append(sns.heatmap(dataFrames[counter] , ax= axs, cmap=colProfile ,center=1, cbar=True, square = True))
            else:
                plots.append(sns.heatmap(dataFrames[counter] , ax= axs[counter], cmap=colProfile,center=1, cbar=True, square = True))#,linewidths= 4,linecolor = "black"))
        else:
            plots.append(sns.heatmap(dataFrames[counter] , ax= axs[int(counter / cols), counter % cols], cmap=colProfile,center=1, cbar=True, square = False)) #
            
            

        plots[counter].set_yticklabels(plots[counter].get_ymajorticklabels(), fontsize = 130, rotation=0)
        plots[counter].set_xticklabels(plots[counter].get_xmajorticklabels(), fontsize = 130, rotation=0)
        cax = plots[counter].figure.axes[-1]
        cax.tick_params(labelsize = 130)
        plots[counter].set_title(""+sampleNames[counter], fontsize = 200)
        
    #fig.delaxes(axs[-1][-1])
    #resolutions = [50,100,150,200]
    resolutions = [150]
    for res in resolutions:
        plt.savefig("C:/Users/KaiKr/Desktop/plots/KCNA2 - fp - 111 - 11s 24s - square - "+str(res)+" dpi.png", dpi=res)

In [25]:
dPath = "C:/Users/KaiKr/Desktop/plots/KCNA2-fp-new.json"
dPath = "C:/Users/KaiKr/Desktop/plots/KCNA2 - fp - 111.json"
with open(dPath) as file:
    data = json.load(file)
    
scanFpFile(data)